# Standard Strategy Optimization Example

This notebook demonstrates parameter optimization for a classic indicator-based strategy using `trade_lab.optimization.OptunaOptimizer`.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'examples' else Path.cwd().resolve()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from trade_lab.backtesting import BacktestEngine
from trade_lab.indicators import EMA, RSI
from trade_lab.optimization import OptunaOptimizer, IntParam, FloatParam
from trade_lab.strategies import StandardStrategy

## 1) Fetch data and split train/validation

In [ ]:
data_engine = BacktestEngine(ticker='^GSPC', start='2015-01-01', end='2025-01-01')
full_df = data_engine.fetch_data()

train_df = full_df[:'2021-12-31']
val_df = full_df['2022-01-01':]

print('Rows total:', len(full_df))
print('Train rows:', len(train_df), '| Val rows:', len(val_df))

## 2) Define strategy factory and search space

In [ ]:
def strategy_factory(params):
    return StandardStrategy(
        indicators=[
            (EMA(period=params['fast']), params['w_fast']),
            (EMA(period=params['slow']), params['w_slow']),
            (RSI(period=params['rsi']), params['w_rsi']),
        ],
        entry_threshold=params['entry_thr'],
        exit_threshold=0.05,
        allow_long=True,
        allow_short=True,
    )

param_space = [
    IntParam('fast', 5, 40),
    IntParam('slow', 20, 150, step=5),
    IntParam('rsi', 7, 28),
    FloatParam('w_fast', 0.1, 3.0),
    FloatParam('w_slow', -3.0, -0.1),
    FloatParam('w_rsi', 0.1, 2.0),
    FloatParam('entry_thr', 0.1, 0.8),
]

## 3) Run optimization

In [ ]:
optimizer = OptunaOptimizer(
    strategy_factory=strategy_factory,
    param_space=param_space,
    train_df=train_df,
    val_df=val_df,
    metric='sharpe_ratio',
    n_trials=50,
    n_jobs=1,
)

result = optimizer.optimize()
print(result.summary())

In [ ]:
result.trials_df.sort_values('value', ascending=False).head(10)

In [ ]:
print('Best train metric:', result.best_value)
if result.val_metrics is not None:
    print('Validation sharpe_ratio:', result.val_metrics.get('sharpe_ratio'))